# Company workers dataset generator #

### Our assistant returns
 dataset in csv format ###

In [ ]:
# Install required libraries
!pip install -U bitsandbytes>=0.46.1
!pip install accelerate
!pip install hf_xet

In [ ]:
# IMPORTS (uncomment to use in google collab)
#from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM,  BitsAndBytesConfig
import torch
import gradio as gr
import pandas as pd
import os
from dotenv import load_dotenv
import io
import re

In [ ]:
# LOG INTO HUGGING FACE
#hf_token = userdata.get('HF_TOKEN')
load_dotenv(override=True)
hf_token = os.getenv('HF_TOKEN')
login(hf_token, add_to_git_credential=True)
# Check if cuda  is available
assert torch.cuda.is_available(), "CUDA is unavailable"
device = "cuda"

In [ ]:
# Our model for generating dataset
LLAMA = 'meta-llama/Llama-3.1-8B-Instruct'

In [ ]:
# PROMPTS
system_prompt = """
„You generate an artificial dataset based on company info provided by the user.

Dataset requirements:

Features: First_name, Last_name, department, position, seniority (Junior, Mid, Senior), basic_salary (numeric), bonus (numeric).

Rules: Respond just with dataset, don't reply like "Here is your dataset: ... " . No NULL values. Data must be realistic: 'Senior' must earn more than 'Junior' within the same department. Salaries should reflect industry standards for the given department.

Format: CSV format inside a code block.

Quantity: Generate exactly 10 rows of data.

Example:

Question:
About Allegro
For around a quarter of a century, we have been serving consumers and promoting the idea of entrepreneurship in one of the most innovative areas of the economy. We are the go-to online shopping destination for millions of consumers and the biggest e-commerce player of European origin, creating the place to do business for thousands of companies, most of them small and medium enterprises. We create innovations that improve the daily lives of millions of Europeans, allowing them to shop for products they need while saving money and time.

Response:
First_name,Last_name,department,basic_salary,bonus,position,seniority
Krzysztof,Kowalski,Sales,40000,10000,Sales Manager,Senior
Aleksandra,Kaczmarek,Logistics,35000,8000,Logistics Coordinator,Mid
Tomasz,Wójcik,Marketing,45000,12000,Marketing Director,Senior
Anna,Kwiatkowska,HR,30000,6000,HR Specialist,Junior
Michał,Zieliński,IT,55000,15000,IT Manager,Senior
Ewa,Sadowska,Finance,38000,10000,Financial Analyst,Mid
Mariusz,Kosowski,Sales,42000,11000,Sales Representative,Senior
Paulina,Majchrzak,Logistics,32000,7000,Logistics Assistant,Junior
Piotr,Bednarczyk,Marketing,48000,13000,Marketing Specialist,Senior
Julia,Gąsior,HR,29000,5000,HR Assistant,Junior
"""

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(LLAMA,  device_map="auto", quantization_config=quant_config)

def generate(messages, max_new_tokens=512):
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

    csv_match = re.search(r"```(?:csv)?\n(.*?)\n```", response, re.DOTALL)
    csv_text = csv_match.group(1) if csv_match else response

    try:
        df = pd.read_csv(io.StringIO(csv_text.strip()))
        return df
    except Exception as e:
        return pd.DataFrame({"Error": [f"Could not parse CSV: {str(e)}"], "Raw Output": [response[:100]]})

In [ ]:
# GRADIO UI
def handle_interaction(message):
    messages = [{"role":"system", "content":system_prompt},{"role":"user", "content":message}]
    return generate(messages)

with gr.Blocks() as ui:
    gr.Markdown("### AI Company Dataset Generator")
    with gr.Row():
        output_df = gr.Dataframe(label="Generated Dataset")
    with gr.Row():
        message = gr.Textbox(label="Enter details about the company (e.g. 'Software house in Poland'):")
        btn = gr.Button("Generate")

    message.submit(handle_interaction, inputs=[message], outputs=[output_df])
    btn.click(handle_interaction, inputs=[message], outputs=[output_df])

ui.launch(debug=True, auth=('eryk', 'eryk'), inbrowser=True)